- Tên công việc: Tính P-RQ3 và đưa DropoutNet vào họ kiểm định thống kê
- Mã/ Mục đối chiếu: W4, V1, V2 (Mục 2.1)
- Code tham khảo trong phụ lục: A.2, A.3

In [1]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/datasets/ngocccchannn/cold-scenario-data/test_cold.parquet
/kaggle/input/datasets/ngocccchannn/cold-scenario-data/val_cold.parquet
/kaggle/input/datasets/ngocccchannn/cold-scenario-data/train_cold.parquet
/kaggle/input/datasets/ngocccchannn/llm-embeddings/llm_profile_embeddings.npy
/kaggle/input/datasets/ngocccchannn/llm-embeddings/full_3769_profiles.json
/kaggle/input/datasets/ngocccchannn/lightgcn-backbone-cold/user_index_cold.json
/kaggle/input/datasets/ngocccchannn/lightgcn-backbone-cold/g_u.npy
/kaggle/input/datasets/ngocccchannn/lightgcn-backbone-cold/g_i.npy
/kaggle/input/datasets/ngocccchannn/lightgcn-backbone-cold/item_index.json
/kaggle/input/datasets/ngocccchannn/cold-scenario/cold_items_test.json
/kaggle/input/datasets/ngocccchannn/cold-scenario/cold_items_val.json
/kaggle/input/datasets/ngocccchannn/evaluator/evaluator.py
/kaggle/input/datasets/ngocccchannn/results-dropoutnet/item_reps/v_e_llm_seed999.npy
/kaggle/input/datasets/ngocccchannn/results-dropoutnet

In [2]:
import copy
import hashlib
import json
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import sys

# Set seed cố định
def set_seed(s=42):
  random.seed(s)
  np.random.seed(s)
  torch.manual_seed(s)
  torch.cuda.manual_seed_all(s)
set_seed(42)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Sử dụng thiết bị: {DEVICE}")
print("[Info] PyTorch:", torch.__version__)
print("[Info] CUDA khả dụng:", torch.cuda.is_available())

sys.path.append('/kaggle/input/datasets/ngocccchannn/evaluator')
from evaluator import evaluate, topk_random_ties

Sử dụng thiết bị: cuda
[Info] PyTorch: 2.11.0+cu128
[Info] CUDA khả dụng: True


In [3]:
# Nạp index và kịch bản cold-start
ITEM_IDX_PATH  = "/kaggle/input/datasets/ngocccchannn/lightgcn-backbone-cold/item_index.json"
USER_IDX_PATH  = "/kaggle/input/datasets/ngocccchannn/lightgcn-backbone-cold/user_index_cold.json"

LLM_IDX_PATH   = "/kaggle/input/datasets/ngocccchannn/embeddings-course-index/embeddings_course_index.json"

TRAIN_PARQUET  = "/kaggle/input/datasets/ngocccchannn/cold-scenario-data/train_cold.parquet"
VAL_PARQUET    = "/kaggle/input/datasets/ngocccchannn/cold-scenario-data/val_cold.parquet"
TEST_PARQUET   = "/kaggle/input/datasets/ngocccchannn/cold-scenario-data/test_cold.parquet"

COLD_VAL_JSON  = "/kaggle/input/datasets/ngocccchannn/cold-scenario/cold_items_val.json"
COLD_TEST_JSON = "/kaggle/input/datasets/ngocccchannn/cold-scenario/cold_items_test.json"

course_ids = json.load(open(ITEM_IDX_PATH))
users      = json.load(open(USER_IDX_PATH))
llm_meta   = json.load(open(LLM_IDX_PATH))

item2idx = {str(c): i for i, c in enumerate(course_ids)}
user2idx = {str(u): i for i, u in enumerate(users)}

cold_val_ids  = json.load(open(COLD_VAL_JSON))
cold_test_ids = json.load(open(COLD_TEST_JSON))

# Hàm chuyển đổi id thành mặt nạ bool
def to_bool_mask(ids, c2i, n_items: int) -> torch.Tensor:
    m = torch.zeros(n_items, dtype=torch.bool)
    m[[c2i[str(c)] for c in ids]] = True
    return m

# Tách nhóm khóa cold-test dựa trên khóa ấm có tương tác trong train_cold (124 trùng / 378 mới)
tr_df = pd.read_parquet(TRAIN_PARQUET)
te_df = pd.read_parquet(TEST_PARQUET)
va_df = pd.read_parquet(VAL_PARQUET)

i_tr = tr_df["course_id"].astype(str).map(item2idx).to_numpy()
warm_items_np = np.unique(i_tr)

_, dup_int = np.unique(llm_meta["dup_group_id"], return_inverse=True)
warm_dup_groups = set(dup_int[warm_items_np].tolist())

cold_test_dup_ids = [c for c in cold_test_ids if dup_int[item2idx[str(c)]] in warm_dup_groups]
cold_test_new_ids = [c for c in cold_test_ids if dup_int[item2idx[str(c)]] not in warm_dup_groups]
assert len(cold_test_dup_ids) == 124 and len(cold_test_new_ids) == 378, "Sai số lượng 124/378 khóa ứng viên!"

cand_test_dup = to_bool_mask(cold_test_dup_ids, item2idx, 3769)
cand_test_new = to_bool_mask(cold_test_new_ids, item2idx, 3769)
cand_bool_mask_a = to_bool_mask(cold_test_ids, item2idx, 3769)

# Định nghĩa cấu trúc groups_prq3 đúng chuẩn từ báo cáo
groups_prq3 = {
    "trung": cand_test_dup,
    "moi": cand_test_new
}

print(f"Đã thiết lập xong groups_prq3 với {len(cold_test_dup_ids)} khóa trùng và {len(cold_test_new_ids)} khóa mới.")

Đã thiết lập xong groups_prq3 với 124 khóa trùng và 378 khóa mới.


In [4]:
def build_hist(df):
    hist_dict = {}
    for u, group in df.groupby('user_id'):
        u_str = str(u)
        if u_str in user2idx:
            u_idx = user2idx[u_str]
            item_idxs = [item2idx[str(i)] for i in group['course_id'] if str(i) in item2idx]
            if len(item_idxs) > 0:
                hist_dict[u_idx] = item_idxs
    return hist_dict

test_target_dict = build_hist(te_df)
print(f'test_target_dict với {len(test_target_dict)} học viên.')

hist_trva = build_hist(pd.concat([tr_df, va_df]))
print(f'hist_trva (train + val) với {len(hist_trva)} học viên.')

test_target_dict với 3340 học viên.
hist_trva (train + val) với 98220 học viên.


In [5]:
@torch.no_grad()
def eval_subgroups(
    score_fn, targets, mask, cand, groups, ks=(5, 10, 20), tie_seed=0, bs=1024):
  """Xếp hạng trên TOÀN BỘ tập ứng viên cand (502 khóa cold-test).

  Với mỗi nhóm, chỉ số chỉ tính trên các đích thuộc nhóm, và chỉ cho học viên có
  ít nhất một đích trong nhóm.
  """
  users_sorted, K = sorted(targets), max(ks)
  g = torch.Generator().manual_seed(tie_seed)
  disc = 1.0 / np.log2(np.arange(2, K + 2))
  cand = torch.as_tensor(cand)
  out = {
      n: {f"{m}@{k}": [] for m in ("Recall", "NDCG", "HR") for k in ks}
      for n in groups
  }
  for s in range(0, len(users_sorted), bs):
    ub = users_sorted[s : s + bs]
    sc = score_fn(torch.tensor(ub)).float().cpu().clone()
    sc[:, ~cand] = -float("inf")
    for r, u in enumerate(ub):
      if mask.get(u):
        sc[r, list(mask[u])] = -float("inf")
    top = topk_random_ties(sc, K, g)  # cùng cách phá hòa với evaluate()
    for r, u in enumerate(ub):
      for n, gset in groups.items():
        #gt = [i for i in targets[u] if i in gset]
        gt = [i for i in targets[u] if gset[i]]
        if not gt:
          continue
        hit = np.isin(top[r], gt).astype(float)
        for k in ks:
          h = hit[:k]
          idcg = disc[: min(len(gt), k)].sum()
          out[n][f"Recall@{k}"].append(h.sum() / len(gt))
          out[n][f"NDCG@{k}"].append((h * disc[:k]).sum() / idcg)
          out[n][f"HR@{k}"].append(float(h.sum() > 0))
  return {n: {m: np.array(v) for m, v in d.items()} for n, d in out.items()}

In [6]:
os.makedirs("scores", exist_ok=True)
seed_list = [42, 2023, 2024, 1007, 999]
model_names = ["raw", "llm", "fused"]
REP_DIR = "/kaggle/input/datasets/ngocccchannn/results-dropoutnet/item_reps"

for name in model_names:
    print(f"\n==================================================================")
    print(f"       ĐÁNH GIÁ P-RQ3 & LƯU SCORES CHO MÔ HÌNH: {name.upper()}")
    print(f"==================================================================")
    
    # Khởi tạo bộ tích lũy cho báo cáo in ra màn hình
    metrics_accumulated = {
        "trung": {m: [] for m in ["Recall@5", "Recall@10", "Recall@20", 
                                   "NDCG@5", "NDCG@10", "NDCG@20", 
                                   "HR@5", "HR@10", "HR@20"]},
        "moi": {m: [] for m in ["Recall@5", "Recall@10", "Recall@20", 
                                 "NDCG@5", "NDCG@10", "NDCG@20", 
                                 "HR@5", "HR@10", "HR@20"]}
    }
    
    # Khởi tạo danh sách tích lũy mảng NDCG@20 từng seed để lưu tệp .npy
    per_seed_ndcg20 = {"trung": [], "moi": []}
    user_counts = {"trung": 0, "moi": 0}
    
    for s in seed_list:
        u_path = f"{REP_DIR}/u_e_{name}_seed{s}.npy"
        v_path = f"{REP_DIR}/v_e_{name}_seed{s}.npy"
        if not os.path.exists(u_path) or not os.path.exists(v_path):
            continue
            
        u_e = torch.tensor(np.load(u_path), device=DEVICE)
        v_e = torch.tensor(np.load(v_path), device=DEVICE)
        fn = lambda ub: u_e[ub.to(DEVICE)] @ v_e.t()
        
        # Chỉ gọi eval_subgroups 1 lần duy nhất cho mỗi seed
        res_sub = eval_subgroups(
            score_fn=fn, 
            targets=test_target_dict, 
            mask=hist_trva, 
            cand=cand_bool_mask_a,
            groups=groups_prq3, 
            ks=(5, 10, 20), 
            tie_seed=s, 
            bs=1024
        )
        
        for g_name in ["trung", "moi"]:
            user_counts[g_name] = len(res_sub[g_name]["NDCG@20"])
            # Lưu mảng NDCG@20 của seed này phục vụ cho việc tính mean .npy sau đó
            per_seed_ndcg20[g_name].append(res_sub[g_name]["NDCG@20"])
            # Tích lũy toàn bộ các metric để in báo cáo
            for m_key in metrics_accumulated[g_name].keys():
                metrics_accumulated[g_name][m_key].append(res_sub[g_name][m_key])
                
    # 1. In kết quả định dạng mean ± std qua 5 seed
    for g_name in ["trung", "moi"]:
        print(f"\n  [Nhóm khóa: {g_name.upper()}] (Số lượng học viên: {user_counts[g_name]})")
        for m_prefix in ["Recall", "NDCG", "HR"]:
            metrics_str_parts = []
            for k in [5, 10, 20]:
                m_key = f"{m_prefix}@{k}"
                seed_means = np.array([np.mean(arr) for arr in metrics_accumulated[g_name][m_key]])
                mean_val = seed_means.mean()
                std_val = seed_means.std(ddof=1) if len(seed_means) > 1 else 0.0
                metrics_str_parts.append(f"{m_key}: {mean_val:.4f} ± {std_val:.4f}")
            print(f"    " + " | ".join(metrics_str_parts))
            
    # 2. Tính trung bình qua 5 seed và lưu tệp điểm NDCG@20 vào thư mục /scores
    print(f"\n  [Lưu tệp kiểm định thống kê]:")
    for g, arrs in per_seed_ndcg20.items():
        if arrs:
            mean_arr = np.mean(np.stack(arrs), axis=0)
            file_path = f"scores/user_ndcg20_dropoutnet_{name}_{g}.npy"
            np.save(file_path, mean_arr)
            print(f"    -> Đã lưu thành công: {file_path} (Kích thước: {mean_arr.shape})")
print("Đã tính và lưu thành công các file điểm P-RQ3 cho DropoutNet vào thư mục scores/")


       ĐÁNH GIÁ P-RQ3 & LƯU SCORES CHO MÔ HÌNH: RAW

  [Nhóm khóa: TRUNG] (Số lượng học viên: 1028)
    Recall@5: 0.0764 ± 0.0071 | Recall@10: 0.1371 ± 0.0032 | Recall@20: 0.2282 ± 0.0172
    NDCG@5: 0.0479 ± 0.0048 | NDCG@10: 0.0676 ± 0.0021 | NDCG@20: 0.0906 ± 0.0050
    HR@5: 0.0794 ± 0.0074 | HR@10: 0.1403 ± 0.0029 | HR@20: 0.2337 ± 0.0206

  [Nhóm khóa: MOI] (Số lượng học viên: 2844)
    Recall@5: 0.0712 ± 0.0113 | Recall@10: 0.1069 ± 0.0062 | Recall@20: 0.1543 ± 0.0067
    NDCG@5: 0.0443 ± 0.0057 | NDCG@10: 0.0564 ± 0.0030 | NDCG@20: 0.0690 ± 0.0033
    HR@5: 0.0779 ± 0.0096 | HR@10: 0.1172 ± 0.0081 | HR@20: 0.1737 ± 0.0093

  [Lưu tệp kiểm định thống kê]:
    -> Đã lưu thành công: scores/user_ndcg20_dropoutnet_raw_trung.npy (Kích thước: (1028,))
    -> Đã lưu thành công: scores/user_ndcg20_dropoutnet_raw_moi.npy (Kích thước: (2844,))

       ĐÁNH GIÁ P-RQ3 & LƯU SCORES CHO MÔ HÌNH: LLM

  [Nhóm khóa: TRUNG] (Số lượng học viên: 1028)
    Recall@5: 0.0509 ± 0.0111 | Recall@10: 0.

In [7]:
import os
import numpy as np

# Đường dẫn tới một file điểm cần kiểm tra
file_path = "/kaggle/working/scores/user_ndcg20_dropoutnet_raw_trung.npy"

if os.path.exists(file_path):
    data = np.load(file_path)
    print(f"--- THÔNG TIN TỆP: {file_path} ---")
    print(f"Kích thước (Shape): {data.shape}")  # Kỳ vọng mong đợi: (1028,) đối với nhóm trùng
    print(f"Kiểu dữ liệu (Dtype): {data.dtype}")
    print(f"Giá trị NDCG@20 trung bình của nhóm: {np.mean(data):.4f}")
    print(f"5 giá trị điểm NDCG@20 đầu tiên của học viên: {data[:5]}")
else:
    print(f"Không tìm thấy tệp tại đường dẫn: {file_path}. Vui lòng kiểm tra lại quá trình chạy cell lưu file.")

--- THÔNG TIN TỆP: /kaggle/working/scores/user_ndcg20_dropoutnet_raw_trung.npy ---
Kích thước (Shape): (1028,)
Kiểu dữ liệu (Dtype): float64
Giá trị NDCG@20 trung bình của nhóm: 0.0906
5 giá trị điểm NDCG@20 đầu tiên của học viên: [0.24532746 0.58613531 0.         0.27034158 0.        ]
